# Financial News Sentiment Analysis Pipeline
## Using FinBERT for Stock Price Movement Prediction

This notebook demonstrates the complete pipeline for:
1. Collecting financial news and stock data
2. Analyzing sentiment using FinBERT
3. Training prediction models
4. Evaluating and visualizing results

**Dataset:** US Stock Market Data with Synthetic Financial News

## Section 1: Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add src directory to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Import project modules
from config.config import PROCESSED_DATA_DIR, MODELS_DIR, RESULTS_DIR, DEVICE
from src.data_pipeline import FinancialDataPipeline
from src.prediction_model import PricePredictionModel, EnsemblePredictor
from src.sentiment_analyzer import FinBERTSentimentAnalyzer
from src.utils import log_message

print("✓ All imports successful!")
print(f"Project root: {project_root}")
print(f"Python version: {sys.version}")

## Section 2: Data Collection and Pipeline Execution

In [ ]:
# Initialize and run data pipeline
print("Starting Data Collection Pipeline...")
print("="*80)

# Create pipeline for Apple stock
pipeline = FinancialDataPipeline(tickers=['AAPL'], days_back=365)

# Run pipeline and get processed data
processed_data = pipeline.run_pipeline(save_intermediate=True)

if processed_data:
    data = processed_data['AAPL']
    print(f"\n✓ Data collected: {len(data)} records")
    print(f"\nColumn names:\n{data.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(data.head())
else:
    print("Error: No data collected")

## Section 3: Data Exploration and Visualization

In [ ]:
# Data statistics
print("Data Statistics:")
print("="*80)
print(data.describe())

print("\n\nSentiment Score Statistics:")
print("="*80)
print(f"Mean Sentiment: {data['daily_sentiment_score'].mean():.4f}")
print(f"Std Sentiment: {data['daily_sentiment_score'].std():.4f}")
print(f"News Count range: {data['news_count'].min():.0f} - {data['news_count'].max():.0f}")

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Stock price
close_col = next(c for c in data.columns if 'close' in str(c).lower())
volume_col = next(c for c in data.columns if 'volume' in str(c).lower() and 'ratio' not in str(c).lower())
axes[0, 0].plot(data['date'], data[close_col], label='Close')
axes[0, 0].set_title('Stock Price Over Time')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].legend()

# Returns distribution
axes[0, 1].hist(data['Returns'].dropna(), bins=50, edgecolor='black')
axes[0, 1].set_title('Returns Distribution')
axes[0, 1].set_xlabel('Returns')

# Sentiment score
axes[1, 0].plot(data['date'], data['daily_sentiment_score'], label='Sentiment', color='green')
axes[1, 0].set_title('Daily Sentiment Score')
axes[1, 0].set_ylabel('Sentiment')
axes[1, 0].legend()

# Volume
axes[1, 1].bar(data['date'], data[volume_col], alpha=0.5)
axes[1, 1].set_title('Trading Volume')
axes[1, 1].set_ylabel('Volume')

plt.tight_layout()
plt.show()

print("✓ Data exploration complete")

## Section 4: Train Prediction Models

In [ ]:
# Train single model first
print("Training Individual Models...")
print("="*80)

# Random Forest
rf_model = PricePredictionModel(model_type='random_forest')
rf_results = rf_model.train(data, test_size=0.2)

# Gradient Boosting
gb_model = PricePredictionModel(model_type='gradient_boosting')
gb_results = gb_model.train(data, test_size=0.2)

print("\n✓ Individual models trained")

In [ ]:
# Train ensemble model
print("\n\nTraining Ensemble Model...")
print("="*80)

ensemble = EnsemblePredictor()
ensemble_results = ensemble.train_all(data, test_size=0.2)

# Compare models
print("\n\nModel Comparison:")
print("="*80)
comparison_data = {
    'Model': ['Random Forest', 'Gradient Boosting'],
    'Test Accuracy': [
        rf_results['metrics']['test']['accuracy'],
        gb_results['metrics']['test']['accuracy']
    ],
    'Test F1-Score': [
        rf_results['metrics']['test']['f1'],
        gb_results['metrics']['test']['f1']
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df)
print("\n✓ All models trained and compared")

## Section 5: Sentiment Analysis with FinBERT

In [ ]:
# Initialize FinBERT sentiment analyzer
print("Initializing FinBERT Sentiment Analyzer...")
analyzer = FinBERTSentimentAnalyzer(device=DEVICE)

# Sample financial headlines
headlines = [
    "Strong quarterly earnings beat analyst expectations",
    "Company announces strategic partnership with industry leader",
    "Stock price drops after disappointing revenue guidance",
    "New product launch exceeds sales targets",
    "Regulatory investigation launched into company practices",
    "Market share gains accelerate in key segments",
    "CEO departure sparks investor concerns",
    "Dividend increased by 15% for the year"
]

# Analyze sentiments
print("\nAnalyzing Sentiments:")
print("="*80)
sentiment_results = []

for headline in headlines:
    result = analyzer.analyze_sentiment(headline, return_scores=True)
    sentiment_results.append(result)
    print(f"'{headline}'")
    print(f"  → Sentiment: {result['sentiment'].upper()}")
    print(f"  → Scores: {result['scores']}\n")

# Convert to DataFrame
sentiment_df = pd.DataFrame(sentiment_results)
print("✓ Sentiment analysis complete")

## Section 6: Results Summary and Visualization

In [ ]:
# Feature importance visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# RF Feature Importance
rf_importance = rf_results['feature_importance'].head(10)
axes[0].barh(rf_importance['feature'], rf_importance['importance'])
axes[0].set_title('Random Forest - Top 10 Feature Importance')
axes[0].set_xlabel('Importance')

# GB Feature Importance
gb_importance = gb_results['feature_importance'].head(10)
axes[1].barh(gb_importance['feature'], gb_importance['importance'])
axes[1].set_title('Gradient Boosting - Top 10 Feature Importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print("✓ Visualization complete")

# Final summary
print("\n" + "="*80)
print("PIPELINE EXECUTION SUMMARY")
print("="*80)
close_col = next(c for c in data.columns if 'close' in str(c).lower())
print(f"\nData Statistics:")
print(f"  - Total records: {len(data)}")
print(f"  - Date range: {data['date'].min()} to {data['date'].max()}")
print(f"  - Average stock price: ${data[close_col].mean():.2f}")
print(f"  - Average sentiment: {data['daily_sentiment_score'].mean():.4f}")

print(f"\nModel Performance (Test Set):")
print(f"  Random Forest:")
print(f"    - Accuracy: {rf_results['metrics']['test']['accuracy']:.4f}")
print(f"    - F1-Score: {rf_results['metrics']['test']['f1']:.4f}")
print(f"\n  Gradient Boosting:")
print(f"    - Accuracy: {gb_results['metrics']['test']['accuracy']:.4f}")
print(f"    - F1-Score: {gb_results['metrics']['test']['f1']:.4f}")

print(f"\nTop Predictive Features:")
print(f"  {rf_results['feature_importance'].head(3).to_string()}")

print("\n" + "="*80)
print("✓ Complete Financial News Sentiment Analysis Pipeline Executed Successfully!")
print("="*80)